# 02 - SOL calibration and error terms

Use `rfmeasurement` to represent and validate measurements; the SOL calculation remains local because it is not yet part of the public API.

In [ ]:
from pathlib import Path
import re
import numpy as np
import skrf as rf
from rfmeasurement.domain import Measurement, MeasurementContext
from rfmeasurement.validation import validate

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
MEASURES_ROOT = ROOT / 'data' / 'calibration'
OUTPUT = ROOT / 'results' / 'error_terms_SOL_master.npz'
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

def natural_key(path):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r'(\d+)', path.stem)]

def files_for(standard):
    folder = MEASURES_ROOT / standard
    s2p = {p.stem: p for p in folder.glob('*.s2p')}
    s1p = {p.stem: p for p in folder.glob('*.s1p')}
    return [s2p[stem] if stem in s2p else s1p[stem] for stem in sorted(set(s2p) | set(s1p), key=lambda x: natural_key(Path(x)))]

def load_and_validate(path, standard):
    network = rf.Network(path)
    measurement = Measurement(
        data=network,
        context=MeasurementContext(
            dut=f'{standard}:{path.stem}',
            instrument='NanoVNA',
            calibration='raw',
        ),
    )
    report = validate(measurement)
    integrity_failures = [
        result for result in report.results
        if result.status.value == 'fail' and result.rule_id.startswith('integrity.')
    ]
    if integrity_failures:
        details = '; '.join(f'{result.rule_id}: {result.explanation}' for result in integrity_failures)
        raise ValueError(f'Medida invalida: {path} ({details})')
    return network.f, network.s[:, 0, 0]

def average(standard):
    traces = [load_and_validate(path, standard) for path in files_for(standard)]
    frequencies = traces[0][0]
    if not all(np.allclose(frequencies, freq) for freq, _ in traces[1:]):
        raise ValueError(f'Rejilla incompatible en {standard}')
    values = np.vstack([gamma for _, gamma in traces])
    return frequencies, values.mean(axis=0), values.std(axis=0)

def estimate_sol(frequencies, open_m, short_m, load_m):
    standards = np.array([1 + 0j, -1 + 0j, 0 + 0j])
    ed = np.empty(len(frequencies), complex); er = np.empty_like(ed); es = np.empty_like(ed)
    for index in range(len(frequencies)):
        measured = np.array([open_m[index], short_m[index], load_m[index]])
        matrix = np.column_stack([np.ones(3), standards, measured * standards])
        ed[index], er[index], es[index] = np.linalg.solve(matrix, measured)
    return ed, er, es

In [4]:
f_open, open_m, open_std = average('open')
f_short, short_m, short_std = average('short')
f_load, load_m, load_std = average('load')
if not (np.allclose(f_open, f_short) and np.allclose(f_open, f_load)):
    raise ValueError('Las medidas SOL no comparten frecuencia')
ed, er, es = estimate_sol(f_open, open_m, short_m, load_m)
np.savez(OUTPUT, freqs_hz=f_open, ed=ed, er=er, es=es,
         sigma_open=open_std, sigma_short=short_std, sigma_load=load_std)
print('Guardado:', OUTPUT)
print('Sigma media open/short/load:', *(np.mean(np.abs(x)) for x in (open_std, short_std, load_std)))

Guardado: /Users/telmomm/Library/CloudStorage/OneDrive-Personal/DOCENCIA/INVESTIGACION/NanoVNA Calibration/error_terms_SOL_master.npz
Sigma media open/short/load: 0.0048432929638096076 0.004507116483853617 0.0005430663727863061
